# Language Detection Model Evaluation

In this notebook, three pretrained language-detection models are evaluated
on the same custom dataset.

The models are compared using:

- Accuracy
- Average inference latency
- Model weight size
- Direct local inference cost

All models are evaluated on CPU using the same 30 text samples.


In [1]:
import gc
from time import perf_counter

import pandas as pd
import torch

from sklearn.metrics import accuracy_score
from transformers import pipeline

d:\PythonProjects\spam-classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
evaluation_samples = [
    # English
    {
        "text": "The meeting starts at nine tomorrow morning.",
        "expected_language": "en",
    },
    {
        "text": "I enjoy learning how computers solve problems.",
        "expected_language": "en",
    },
    {
        "text": "Please send the final report before Friday.",
        "expected_language": "en",
    },
    {
        "text": "The weather is pleasant and the streets are quiet.",
        "expected_language": "en",
    },
    {
        "text": "This application helps users organize their daily tasks.",
        "expected_language": "en",
    },

    # Arabic
    {
        "text": "يبدأ الاجتماع الساعة التاسعة صباحًا غدًا.",
        "expected_language": "ar",
    },
    {
        "text": "أحب تعلم كيفية عمل نماذج الذكاء الاصطناعي.",
        "expected_language": "ar",
    },
    {
        "text": "من فضلك أرسل التقرير النهائي قبل يوم الجمعة.",
        "expected_language": "ar",
    },
    {
        "text": "الطقس معتدل والشوارع هادئة اليوم.",
        "expected_language": "ar",
    },
    {
        "text": "يساعد هذا التطبيق المستخدمين على تنظيم مهامهم اليومية.",
        "expected_language": "ar",
    },

    # French
    {
        "text": "La réunion commence demain matin à neuf heures.",
        "expected_language": "fr",
    },
    {
        "text": "J'aime apprendre comment fonctionnent les ordinateurs.",
        "expected_language": "fr",
    },
    {
        "text": "Veuillez envoyer le rapport final avant vendredi.",
        "expected_language": "fr",
    },
    {
        "text": "Le temps est agréable et les rues sont calmes.",
        "expected_language": "fr",
    },
    {
        "text": "Cette application aide les utilisateurs à organiser leurs tâches.",
        "expected_language": "fr",
    },

    # German
    {
        "text": "Die Besprechung beginnt morgen früh um neun Uhr.",
        "expected_language": "de",
    },
    {
        "text": "Ich lerne gerne, wie Computer Probleme lösen.",
        "expected_language": "de",
    },
    {
        "text": "Bitte senden Sie den Abschlussbericht vor Freitag.",
        "expected_language": "de",
    },
    {
        "text": "Das Wetter ist angenehm und die Straßen sind ruhig.",
        "expected_language": "de",
    },
    {
        "text": "Diese Anwendung hilft Benutzern bei der Organisation ihrer Aufgaben.",
        "expected_language": "de",
    },

    # Spanish
    {
        "text": "La reunión comienza mañana a las nueve de la mañana.",
        "expected_language": "es",
    },
    {
        "text": "Me gusta aprender cómo los ordenadores resuelven problemas.",
        "expected_language": "es",
    },
    {
        "text": "Por favor, envía el informe final antes del viernes.",
        "expected_language": "es",
    },
    {
        "text": "El clima es agradable y las calles están tranquilas.",
        "expected_language": "es",
    },
    {
        "text": "Esta aplicación ayuda a los usuarios a organizar sus tareas.",
        "expected_language": "es",
    },

    # Italian
    {
        "text": "La riunione inizia domani mattina alle nove.",
        "expected_language": "it",
    },
    {
        "text": "Mi piace imparare come i computer risolvono i problemi.",
        "expected_language": "it",
    },
    {
        "text": "Per favore, invia il rapporto finale prima di venerdì.",
        "expected_language": "it",
    },
    {
        "text": "Il tempo è piacevole e le strade sono tranquille.",
        "expected_language": "it",
    },
    {
        "text": "Questa applicazione aiuta gli utenti a organizzare le loro attività.",
        "expected_language": "it",
    },
]

evaluation_df = pd.DataFrame(evaluation_samples)

print("Number of evaluation samples:", len(evaluation_df))

evaluation_df.head()

Number of evaluation samples: 30


,text,expected_language
0,The meeting starts at nine tomorrow morning.,en
1,I enjoy learning how computers solve problems.,en
2,Please send the final report before Friday.,en
3,The weather is pleasant and the streets are qu...,en
4,This application helps users organize their da...,en


In [3]:
evaluation_df.to_csv(
    "evaluation_data.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Evaluation dataset saved successfully.")

Evaluation dataset saved successfully.


In [4]:
models_info = {
    "Papluca XLM-RoBERTa": {
        "model_id": "papluca/xlm-roberta-base-language-detection",
        "supported_languages": 20,
        "approx_weight_size_gb": 1.11,
        "direct_local_cost_usd": 0.0,
    },

    "JB2K Multilingual BERT": {
        "model_id": (
            "jb2k/"
            "bert-base-multilingual-cased-language-detection"
        ),
        "supported_languages": 45,
        "approx_weight_size_gb": 0.71,
        "direct_local_cost_usd": 0.0,
    },

    "Multilingual E5": {
        "model_id": (
            "Mike0307/"
            "multilingual-e5-language-detection"
        ),
        "supported_languages": 45,
        "approx_weight_size_gb": 1.11,
        "direct_local_cost_usd": 0.0,
    },
}

In [5]:
common_language_codes = [
    "ar",      # LABEL_0  - Arabic
    "eu",      # LABEL_1  - Basque
    "br",      # LABEL_2  - Breton
    "ca",      # LABEL_3  - Catalan
    "zh-cn",   # LABEL_4  - Chinese China
    "zh-hk",   # LABEL_5  - Chinese Hong Kong
    "zh-tw",   # LABEL_6  - Chinese Taiwan
    "cv",      # LABEL_7  - Chuvash
    "cs",      # LABEL_8  - Czech
    "dv",      # LABEL_9  - Dhivehi
    "nl",      # LABEL_10 - Dutch
    "en",      # LABEL_11 - English
    "eo",      # LABEL_12 - Esperanto
    "et",      # LABEL_13 - Estonian
    "fr",      # LABEL_14 - French
    "fy",      # LABEL_15 - Frisian
    "ka",      # LABEL_16 - Georgian
    "de",      # LABEL_17 - German
    "el",      # LABEL_18 - Greek
    "cnh",     # LABEL_19 - Hakha Chin
    "id",      # LABEL_20 - Indonesian
    "ia",      # LABEL_21 - Interlingua
    "it",      # LABEL_22 - Italian
    "ja",      # LABEL_23 - Japanese
    "kab",     # LABEL_24 - Kabyle
    "rw",      # LABEL_25 - Kinyarwanda
    "ky",      # LABEL_26 - Kyrgyz
    "lv",      # LABEL_27 - Latvian
    "mt",      # LABEL_28 - Maltese
    "mn",      # LABEL_29 - Mongolian
    "fa",      # LABEL_30 - Persian
    "pl",      # LABEL_31 - Polish
    "pt",      # LABEL_32 - Portuguese
    "ro",      # LABEL_33 - Romanian
    "rm",      # LABEL_34 - Romansh
    "ru",      # LABEL_35 - Russian
    "sah",     # LABEL_36 - Sakha
    "sl",      # LABEL_37 - Slovenian
    "es",      # LABEL_38 - Spanish
    "sv",      # LABEL_39 - Swedish
    "ta",      # LABEL_40 - Tamil
    "tt",      # LABEL_41 - Tatar
    "tr",      # LABEL_42 - Turkish
    "uk",      # LABEL_43 - Ukrainian
    "cy",      # LABEL_44 - Welsh
]

common_label_to_iso = {
    f"LABEL_{index}": language_code
    for index, language_code in enumerate(common_language_codes)
}

In [6]:
def normalize_language_label(raw_label):
    """
    Convert different model label formats into ISO-like language codes.

    Examples:
    LABEL_11 -> en
    LABEL_0  -> ar
    en       -> en
    """

    cleaned_label = raw_label.strip()

    if cleaned_label in common_label_to_iso:
        return common_label_to_iso[cleaned_label]

    return cleaned_label.lower()

In [7]:
print(normalize_language_label("LABEL_0"))
print(normalize_language_label("LABEL_11"))
print(normalize_language_label("LABEL_14"))
print(normalize_language_label("LABEL_17"))
print(normalize_language_label("LABEL_22"))
print(normalize_language_label("LABEL_38"))
print(normalize_language_label("en"))

ar
en
fr
de
it
es
en


In [8]:
def evaluate_model(model_name, model_info, evaluation_data):
    """
    Load one model, evaluate it on all samples,
    and calculate accuracy and average latency.
    """

    model_id = model_info["model_id"]

    print("=" * 70)
    print(f"Loading model: {model_name}")
    print(f"Model ID: {model_id}")

    load_start_time = perf_counter()

    classifier = pipeline(
        task="text-classification",
        model=model_id,
        tokenizer=model_id,
        device=-1,
    )

    load_time_seconds = perf_counter() - load_start_time

    print(f"Model loaded in {load_time_seconds:.2f} seconds.")

    # Warm-up prediction
    first_text = evaluation_data.iloc[0]["text"]

    classifier(
        first_text,
        truncation=True,
    )

    predictions = []
    confidence_scores = []
    latency_values = []
    raw_labels = []

    for row in evaluation_data.itertuples(index=False):
        prediction_start_time = perf_counter()

        output = classifier(
            row.text,
            truncation=True,
        )[0]

        prediction_latency = (
            perf_counter() - prediction_start_time
        )

        raw_label = output["label"]

        normalized_label = normalize_language_label(
            raw_label
        )

        predictions.append(normalized_label)
        confidence_scores.append(output["score"])
        latency_values.append(prediction_latency)
        raw_labels.append(raw_label)

    model_accuracy = accuracy_score(
        evaluation_data["expected_language"],
        predictions,
    )

    average_latency = (
        sum(latency_values) / len(latency_values)
    )

    prediction_details = evaluation_data.copy()

    prediction_details["model_name"] = model_name
    prediction_details["raw_label"] = raw_labels
    prediction_details["predicted_language"] = predictions
    prediction_details["confidence_score"] = confidence_scores
    prediction_details["latency_seconds"] = latency_values

    prediction_details["correct"] = (
        prediction_details["expected_language"]
        == prediction_details["predicted_language"]
    )

    summary = {
        "Model": model_name,
        "Accuracy": model_accuracy,
        "Average Latency (seconds)": average_latency,
        "Loading Time (seconds)": load_time_seconds,
        "Supported Languages": model_info[
            "supported_languages"
        ],
        "Approx. Weight Size (GB)": model_info[
            "approx_weight_size_gb"
        ],
        "Direct Local Cost (USD/prediction)": model_info[
            "direct_local_cost_usd"
        ],
    }

    print(f"Accuracy: {model_accuracy:.2%}")
    print(
        "Average latency:",
        f"{average_latency:.4f} seconds per text",
    )

    del classifier
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary, prediction_details

In [9]:
comparison_summaries = []
all_prediction_details = []

for model_name, model_info in models_info.items():
    model_summary, model_predictions = evaluate_model(
        model_name=model_name,
        model_info=model_info,
        evaluation_data=evaluation_df,
    )

    comparison_summaries.append(model_summary)
    all_prediction_details.append(model_predictions)

print("All models were evaluated successfully.")

Loading model: Papluca XLM-RoBERTa
Model ID: papluca/xlm-roberta-base-language-detection


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3745.36it/s]


Model loaded in 5.46 seconds.
Accuracy: 100.00%
Average latency: 0.0463 seconds per text
Loading model: JB2K Multilingual BERT
Model ID: jb2k/bert-base-multilingual-cased-language-detection


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14803.43it/s]


Model loaded in 2.55 seconds.
Accuracy: 100.00%
Average latency: 0.0491 seconds per text
Loading model: Multilingual E5
Model ID: Mike0307/multilingual-e5-language-detection


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15525.59it/s]


Model loaded in 4.33 seconds.
Accuracy: 100.00%
Average latency: 0.0516 seconds per text
All models were evaluated successfully.


In [10]:
comparison_df = pd.DataFrame(
    comparison_summaries
)

comparison_df = comparison_df.sort_values(
    by=[
        "Accuracy",
        "Average Latency (seconds)",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(drop=True)

comparison_df

,Model,Accuracy,Average Latency (seconds),Loading Time (seconds),Supported Languages,Approx. Weight Size (GB),Direct Local Cost (USD/prediction)
0,Papluca XLM-RoBERTa,1.0,0.046321,5.459802,20,1.11,0.0
1,JB2K Multilingual BERT,1.0,0.049120,2.547979,45,0.71,0.0
2,Multilingual E5,1.0,0.051568,4.333388,45,1.11,0.0


In [11]:
comparison_display = comparison_df.copy()

comparison_display["Accuracy"] = (
    comparison_display["Accuracy"] * 100
).round(2).astype(str) + "%"

comparison_display["Average Latency (seconds)"] = (
    comparison_display[
        "Average Latency (seconds)"
    ].round(4)
)

comparison_display["Loading Time (seconds)"] = (
    comparison_display[
        "Loading Time (seconds)"
    ].round(2)
)

comparison_display

,Model,Accuracy,Average Latency (seconds),Loading Time (seconds),Supported Languages,Approx. Weight Size (GB),Direct Local Cost (USD/prediction)
0,Papluca XLM-RoBERTa,100.0%,0.0463,5.46,20,1.11,0.0
1,JB2K Multilingual BERT,100.0%,0.0491,2.55,45,0.71,0.0
2,Multilingual E5,100.0%,0.0516,4.33,45,1.11,0.0


In [12]:
comparison_df.to_csv(
    "model_comparison_results.csv",
    index=False,
)

print("Comparison table saved successfully.")

Comparison table saved successfully.


In [13]:
all_predictions_df = pd.concat(
    all_prediction_details,
    ignore_index=True,
)

all_predictions_df.to_csv(
    "all_model_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

all_predictions_df.head(10)

,text,expected_language,model_name,raw_label,predicted_language,confidence_score,latency_seconds,correct
0,The meeting starts at nine tomorrow morning.,en,Papluca XLM-RoBERTa,en,en,0.953928,0.046486,True
1,I enjoy learning how computers solve problems.,en,Papluca XLM-RoBERTa,en,en,0.985136,0.046654,True
2,Please send the final report before Friday.,en,Papluca XLM-RoBERTa,en,en,0.975997,0.049330,True
3,The weather is pleasant and the streets are qu...,en,Papluca XLM-RoBERTa,en,en,0.993905,0.046501,True
4,This application helps users organize their da...,en,Papluca XLM-RoBERTa,en,en,0.988381,0.044121,True
5,يبدأ الاجتماع الساعة التاسعة صباحًا غدًا.,ar,Papluca XLM-RoBERTa,ar,ar,0.978656,0.047066,True
6,أحب تعلم كيفية عمل نماذج الذكاء الاصطناعي.,ar,Papluca XLM-RoBERTa,ar,ar,0.994098,0.050071,True
7,من فضلك أرسل التقرير النهائي قبل يوم الجمعة.,ar,Papluca XLM-RoBERTa,ar,ar,0.993458,0.048131,True
8,الطقس معتدل والشوارع هادئة اليوم.,ar,Papluca XLM-RoBERTa,ar,ar,0.993539,0.047333,True
9,يساعد هذا التطبيق المستخدمين على تنظيم مهامهم ...,ar,Papluca XLM-RoBERTa,ar,ar,0.993408,0.050487,True


In [14]:
incorrect_predictions_df = all_predictions_df[
    all_predictions_df["correct"] == False
].copy()

incorrect_predictions_df[
    [
        "model_name",
        "text",
        "expected_language",
        "predicted_language",
        "confidence_score",
    ]
]

,model_name,text,expected_language,predicted_language,confidence_score


In [15]:
winner_df = comparison_df.sort_values(
    by=[
        "Accuracy",
        "Average Latency (seconds)",
        "Approx. Weight Size (GB)",
    ],
    ascending=[
        False,
        True,
        True,
    ],
)

winner = winner_df.iloc[0]

print("Selected winner:", winner["Model"])
print(f"Accuracy: {winner['Accuracy']:.2%}")
print(
    "Average latency:",
    f"{winner['Average Latency (seconds)']:.4f} seconds",
)
print(
    "Approximate weight size:",
    f"{winner['Approx. Weight Size (GB)']:.2f} GB",
)

Selected winner: Papluca XLM-RoBERTa
Accuracy: 100.00%
Average latency: 0.0463 seconds
Approximate weight size: 1.11 GB
